<a href="https://colab.research.google.com/github/AlfredXNet/Customer-Churn-Prediction-App/blob/main/%F0%9F%93%8A_Customer_Churn_Prediction_Clean_Training_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALL AND IMPORT LIBRARIES

In [1]:
!pip install joblib scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

Load the data

In [2]:
from google.colab import files
uploaded = files.upload()

# Load the data
import io
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

Saving insurance-churn-insights.csv to insurance-churn-insights.csv


In [3]:
# Displaying the shape of the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nChurn distribution:\n{df['churned'].value_counts()}")

Dataset shape: (200, 31)

Columns: ['policyholder_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'phone_number', 'address_street', 'address_city', 'address_state', 'address_postal_code', 'address_country', 'policy_id', 'policyholder_id_ref', 'policy_type', 'policy_start_date', 'policy_end_date', 'policy_status', 'premium_amount', 'payment_frequency', 'total_claims_count', 'total_claims_amount', 'last_claim_date', 'churned', 'churn_date', 'churn_reason', 'customer_tenure_months', 'last_contact_date', 'contact_channel', 'satisfaction_score', 'number_of_policies']

Churn distribution:
churned
False    155
True      45
Name: count, dtype: int64


DROP LEAKAGE AND IRRELEVANT COLUMNS

In [4]:
columns_to_drop = [
    # Identifying information
    'policyholder_id', 'first_name', 'last_name', 'email', 'phone_number',
    'address_street', 'address_city', 'address_state', 'address_postal_code',
    'address_country',

    # Policy identifiers
    'policy_id', 'policyholder_id_ref',

    # DATES - THESE CAUSE LEAKAGE!
    'date_of_birth',           # Age is better than birth date
    'policy_start_date',       # Tenure is better than start date
    'policy_end_date',         # Only available after policy ends
    'last_contact_date',       # Too granular, use contact_channel instead
    'last_claim_date',         # This caused your error!
    'churn_date',              # Only available AFTER churn

    # Post-churn information
    'churn_reason',            # Only available AFTER churn
]

# Only drop columns that exist
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df_clean = df.drop(columns=existing_cols_to_drop)

print(f"After dropping {len(existing_cols_to_drop)} columns: {df_clean.shape}")

After dropping 19 columns: (200, 12)


HANDLE MISSING VALUES

In [6]:
print("\nMissing values before:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Fill numerical missing values with 0 or median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col].fillna(0, inplace=True)

# Fill categorical missing values with 'Unknown'
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().any():
        df_clean[col].fillna('Unknown', inplace=True)

print("\nMissing values after:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])


Missing values before:
total_claims_amount    63
dtype: int64

Missing values after:
Series([], dtype: int64)


IDENTIFY CATEGORICAL COLUMNS FOR ENCODING

In [7]:
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns to encode: {categorical_cols}")

# Remove 'churned' from categorical columns if it's there
if 'churned' in categorical_cols:
    categorical_cols.remove('churned')


Categorical columns to encode: ['gender', 'policy_type', 'policy_status', 'payment_frequency', 'contact_channel']


ONE-HOT ENCODING

In [8]:
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print(f"\nAfter encoding: {df_encoded.shape}")
print(f"Features: {df_encoded.columns.tolist()}")


After encoding: (200, 23)
Features: ['premium_amount', 'total_claims_count', 'total_claims_amount', 'churned', 'customer_tenure_months', 'satisfaction_score', 'number_of_policies', 'age', 'gender_male', 'gender_other', 'gender_prefer_not_to_say', 'policy_type_life', 'policy_status_cancelled', 'policy_status_expired', 'policy_status_lapsed', 'payment_frequency_monthly', 'payment_frequency_quarterly', 'payment_frequency_semi-annual', 'contact_channel_in-person', 'contact_channel_mail', 'contact_channel_online-portal', 'contact_channel_other', 'contact_channel_phone']


PREPARE FEATURES AND TARGET

In [9]:
# Ensure churned is boolean/int
if df_encoded['churned'].dtype == 'bool':
    df_encoded['churned'] = df_encoded['churned'].astype(int)

X = df_encoded.drop('churned', axis=1)
y = df_encoded['churned']

print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Save feature names for later use in the app
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'training_features.pkl')
print(f"✅ Saved {len(feature_names)} feature names")


Features shape: (200, 22)
Target distribution:
churned
0    155
1     45
Name: count, dtype: int64
✅ Saved 22 feature names


TRAIN/TEST SPLIT

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Training set: (160, 22)
Test set: (40, 22)


SCALE FEATURES

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, 'scaler.pkl')
print("✅ Saved scaler")

✅ Saved scaler


TRAIN MULTIPLE MODELS

In [12]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

# Train and evaluate
results = []
trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

    print(f"\n{name} trained ✓")


Logistic Regression trained ✓

Decision Tree trained ✓

Random Forest trained ✓

SVM trained ✓


DISPLAY RESULTS

In [13]:
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(results_df.to_string(index=False))

# Find best model based on F1 score
best_model_name = results_df.loc[results_df['F1 Score'].idxmax(), 'Model']
best_model = trained_models[best_model_name]

print(f"\n🏆 Best model: {best_model_name}")


MODEL PERFORMANCE
              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression       1.0        1.0     1.0       1.0
      Decision Tree       1.0        1.0     1.0       1.0
      Random Forest       1.0        1.0     1.0       1.0
                SVM       1.0        1.0     1.0       1.0

🏆 Best model: Logistic Regression


 SAVE THE BEST MODEL

In [14]:
joblib.dump(best_model, 'churn_model.pkl')
print(f"✅ Saved best model as 'churn_model.pkl'")

# Also save all models for comparison
for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_') + '_model.pkl'
    joblib.dump(model, filename)
    print(f"✅ Saved {filename}")

✅ Saved best model as 'churn_model.pkl'
✅ Saved logistic_regression_model.pkl
✅ Saved decision_tree_model.pkl
✅ Saved random_forest_model.pkl
✅ Saved svm_model.pkl


DETAILED EVALUATION OF BEST MODEL

In [15]:
print("\n" + "="*50)
print(f"DETAILED EVALUATION - {best_model_name}")
print("="*50)

# Classification report
y_pred_best = best_model.predict(X_test_scaled)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, columns=['Predicted 0', 'Predicted 1'], index=['Actual 0', 'Actual 1']))


DETAILED EVALUATION - Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00         9

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40


Confusion Matrix:
          Predicted 0  Predicted 1
Actual 0           31            0
Actual 1            0            9


DOWNLOAD ALL FILES

In [16]:
print("\n" + "="*50)
print("DOWNLOADING FILES")
print("="*50)

# List all files to download
files_to_download = [
    'churn_model.pkl',
    'scaler.pkl',
    'training_features.pkl',
    'logistic_regression_model.pkl',
    'decision_tree_model.pkl',
    'random_forest_model.pkl',
    'svm_model.pkl'
]

for file in files_to_download:
    try:
        files.download(file)
        print(f"📥 Downloaded: {file}")
    except:
        print(f"⚠️ Could not download: {file}")

print("\n✅ All done! Place these files in your Streamlit app folder.")


DOWNLOADING FILES


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloaded: churn_model.pkl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

📥 Downloaded: random_forest_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloaded: svm_model.pkl

✅ All done! Place these files in your Streamlit app folder.


In [20]:
# Train Logistic Regression with stronger regularization
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib

# Scale your features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train with stronger regularization (C=0.1 means more regularization)
model = LogisticRegression(
    C=0.05,  # Stronger regularization (prevents overfitting)
    class_weight='balanced',  # Handles class imbalance
    max_iter=1000,
    random_state=42
)

model.fit(X_scaled, y)

# Save the new model (overwrite the old one)
joblib.dump(model, 'churn_model.pkl')

# Save scaler and feature names (same as before)
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'training_features.pkl')

print("✅ New model saved with stronger regularization!")

✅ New model saved with stronger regularization!


In [21]:
from google.colab import files
files.download('churn_model.pkl')
files.download('scaler.pkl')
files.download('training_features.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>